In [0]:
%sql
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM ecommerce.base.customers
UNION ALL SELECT 'orders', COUNT(*) FROM ecommerce.base.orders
UNION ALL SELECT 'order_items', COUNT(*) FROM ecommerce.base.order_items
UNION ALL SELECT 'products', COUNT(*) FROM ecommerce.base.products
UNION ALL SELECT 'payments', COUNT(*) FROM ecommerce.base.payments
UNION ALL SELECT 'returns', COUNT(*) FROM ecommerce.base.returns
UNION ALL SELECT 'regions', COUNT(*) FROM ecommerce.base.regions
UNION ALL SELECT 'stores', COUNT(*) FROM ecommerce.base.stores
UNION ALL SELECT 'categories', COUNT(*) FROM ecommerce.base.categories;

table_name,row_count
customers,15045
orders,120000
order_items,260932
products,500
payments,120000
returns,15050
regions,8
stores,20
categories,12


In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT order_id, COUNT(*) 
FROM ecommerce.base.orders 
GROUP BY order_id 
HAVING COUNT(*) > 1;

order_id,COUNT(*)


In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT category_id, COUNT(*) 
FROM ecommerce.base.categories 
GROUP BY category_id 
HAVING COUNT(*) > 1;

category_id,COUNT(*)


In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT customer_id, COUNT(*) 
FROM ecommerce.base.customers 
GROUP BY customer_id 
HAVING COUNT(*) > 1;

customer_id,COUNT(*)


In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT product_id, COUNT(*) 
FROM ecommerce.base.products 
GROUP BY product_id 
HAVING COUNT(*) > 1; 

product_id,COUNT(*)


In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT payment_id, COUNT(*) 
FROM ecommerce.base.payments 
GROUP BY payment_id 
HAVING COUNT(*) > 1; 

payment_id,COUNT(*)


In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT region_id, COUNT(*) 
FROM ecommerce.base.regions 
GROUP BY region_id 
HAVING COUNT(*) > 1; 

region_id,COUNT(*)


In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT return_id, COUNT(*) 
FROM ecommerce.base.returns 
GROUP BY return_id 
HAVING COUNT(*) > 1; 

return_id,COUNT(*)


In [0]:
%sql
-- Primary key uniqueness check (should return 0 rows if clean)
SELECT store_id, COUNT(*) 
FROM ecommerce.base.stores 
GROUP BY store_id 
HAVING COUNT(*) > 1; 


store_id,COUNT(*)


In [0]:
%sql
-- Orphan check: orders referencing a customer_id that doesn't exist in customers
SELECT o.customer_id
FROM ecommerce.base.orders o
LEFT JOIN ecommerce.base.customers c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

customer_id


In [0]:
%sql
-- Orphan check: orders referencing a region_id that doesn't exist in regions
SELECT o.region_id
FROM ecommerce.base.orders o
LEFT JOIN ecommerce.base.regions r ON o.region_id = r.region_id
WHERE r.region_id IS NULL;

region_id


In [0]:
%sql
-- Orphan check: orders referencing a store_id that doesn't exist in stores
SELECT o.store_id
FROM ecommerce.base.orders o
LEFT JOIN ecommerce.base.stores s ON o.store_id = s.store_id
WHERE s.store_id IS NULL;

store_id


In [0]:
%sql
-- Orphan check: products referencing a category_id that doesn't exist in categories
SELECT p.category_id
FROM ecommerce.base.products p
LEFT JOIN ecommerce.base.categories c ON p.category_id = c.category_id
WHERE c.category_id IS NULL;


category_id


In [0]:
%sql

-- Orphan check: payments referencing an order_id that doesn't exist in orders
SELECT p.order_id
FROM ecommerce.base.payments p
LEFT JOIN ecommerce.base.orders o ON p.order_id = o.order_id
WHERE o.order_id IS NULL;


order_id


In [0]:
%sql

-- Orphan check: returns referencing an order_id that doesn't exist in orders
SELECT r.order_id
FROM ecommerce.base.returns r
LEFT JOIN ecommerce.base.orders o ON r.order_id = o.order_id
WHERE o.order_id IS NULL;

order_id


In [0]:
%sql
-- Cardinality check: unique counts across key dimension columns
SELECT
  (SELECT COUNT(DISTINCT customer_id) FROM ecommerce.base.customers) AS unique_customers,
  (SELECT COUNT(DISTINCT product_id) FROM ecommerce.base.products) AS unique_products,
  (SELECT COUNT(DISTINCT region_id) FROM ecommerce.base.regions) AS unique_regions,
  (SELECT COUNT(DISTINCT store_id) FROM ecommerce.base.stores) AS unique_stores,
  (SELECT COUNT(DISTINCT category_id) FROM ecommerce.base.categories) AS unique_categories,
  (SELECT COUNT(DISTINCT brand) FROM ecommerce.base.products) AS unique_brands,
  (SELECT COUNT(DISTINCT city) FROM ecommerce.base.customers) AS unique_cities;

unique_customers,unique_products,unique_regions,unique_stores,unique_categories,unique_brands,unique_cities
15045,500,8,20,12,140,10710


In [0]:
%sql
-- Full-row duplicate check: same customer, same order date, same total (likely double-entry)
SELECT customer_id, order_date, order_total, COUNT(*) AS occurrences
FROM ecommerce.base.orders
GROUP BY customer_id, order_date, order_total
HAVING COUNT(*) > 1;

customer_id,order_date,order_total,occurrences
13198,2025-12-31,21.84,2


In [0]:
%sql
-- Full-row duplicate check: same order, same product, same quantity/price repeated
SELECT order_id, product_id, quantity, unit_price, COUNT(*) AS occurrences
FROM ecommerce.base.order_items
GROUP BY order_id, product_id, quantity, unit_price
HAVING COUNT(*) > 1;

order_id,product_id,quantity,unit_price,occurrences
10971,148,4,48.22,2
26928,313,1,34.85,2
58208,313,1,35.23,2
58819,313,3,34.82,2
77464,313,1,35.08,2
84563,100,1,10.56,2
100024,313,2,34.49,2
108624,129,1,24.96,2
109562,129,1,25.03,2


In [0]:
%sql
SELECT order_id, product_id, quantity, unit_price,
       COUNT(*) AS occurrences,
       COUNT(DISTINCT order_item_id) AS unique_items
FROM ecommerce.base.order_items
GROUP BY order_id, product_id, quantity, unit_price
HAVING COUNT(*) > 1;

order_id,product_id,quantity,unit_price,occurrences,unique_items
10971,148,4,48.22,2,2
26928,313,1,34.85,2,2
58208,313,1,35.23,2,2
58819,313,3,34.82,2,2
77464,313,1,35.08,2,2
84563,100,1,10.56,2,2
100024,313,2,34.49,2,2
108624,129,1,24.96,2,2
109562,129,1,25.03,2,2
